In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteriaList
import torch
import re
import os


/home/rebecca.almeida/6G_network_reasoning_model/reasoning_6g/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
os.makedirs("./results", exist_ok=True)
os.makedirs("./figures", exist_ok=True)

### Two-Stage Output Generation

1. **Stage 1 - Reasoning:** the model is prompted to think freely inside a `<think>...</think>` block, producing the full chain-of-thought.
2. **Stage 2 - Answer extraction:** the raw token sequence from stage 1 is truncated right after `</think>` and continued with a forced `"Answer:"` prefix, so the model emits only the selected option letter under greedy decoding.

In [ ]:

# ====== Zero-shot =========
Instructions = f"""You are a network orchestration reasoning agent. You will be given a scenario 
                    describing current network conditions, constraints, and a set of possible 
                    actions. Analyze the trade-offs between options under uncertainty, then select 
                    the option that best satisfies the stated constraints while minimizing 
                    worst-case risk. Answer only with the correct letter. 
                """
# ====== One-shot =========
# Instructions = f"""
#      You are an expert telecommunications and autonomous systems reasoning engine. 
#      Analyze network troubleshooting and slicing decisions step-by-step, 
#      valuating constraints, SLA violations, environmental risk, and minimax regret, 
#      followed by your final choice letter. Be concise. 

#     Example

#     Question:
#     At turn 7, the UAV is operating under mMTC slice with latency 52ms, jitter 15ms, loss 3.1%, throughput 0.9Mbps, and edge load 0.72. The SLA requires latency < 40ms, jitter < 10ms, loss < 1%, and throughput > 1Mbps. Weather is icing with wind 9.5m/s. The UAV must complete thermal inspection in 2 turns or abort. If degradation persists, latency may rise to 60ms next turn and edge load to 0.78. Under worst-case projection, which option minimizes maximum regret over the next 2 turns while preserving mission integrity? 

#     Options:
#     A) Stay on mMTC, disable all sensors except thermal, accept 100% SLA violation risk.
#     B) Switch to URLLC now, accept 15% throughput drop, but guarantee latency < 12ms and jitter < 2ms for next 2 turns.
#     C) Switch to eMBB now, gain 800Mbps throughput but risk latency spiking to 25ms if edge load exceeds 0.75.
#     D) Do nothing, wait for natural recovery; historical data shows 30% chance of improvement by turn 9.

#     <think>
#     Current state (mMTC): Latency 52ms (SLA < 40ms), Jitter 15ms (SLA < 10ms), Loss 3.1% (SLA < 1%), Throughput 0.9Mbps (SLA > 1Mbps).
#     Operational constraint: Thermal inspection must complete in 2 turns (Turns 7–8) under severe icing/wind conditions.
#     Projected worst-case: Edge load rises to 0.78 and latency to 60ms next turn.
#     Option A: Guarantees 100% SLA failure and risks mission abort due to unmitigated latency/loss. High maximum regret.
#     Option B (URLLC): Accepts a minor throughput loss (0.9Mbps - 15% = ~0.765Mbps) but strictly guarantees latency < 12ms and jitter < 2ms regardless of edge load rising to 0.78. Holds latency within bounds to complete the 2-turn window safely.
#     Option C (eMBB): Edge load is projected at 0.78, which exceeds the 0.75 threshold. This triggers an automatic latency spike to 25ms and higher loss risk in icing conditions, violating worst-case safety stability.
#     Option D: Relies on a 30% probability of recovery, leaving a 70% risk of catastrophic failure during a time-sensitive 2-turn deadline. High maximum regret.
#     The maximum regret of failing the mission due to control/latency failure vastly outweighs a temporary throughput reduction. Option B eliminates bounded latency risks during the critical 2-turn window, making it the minimax regret choice.
#     </think>

#     Answer: B
#     """

In [ ]:
from utils.token_utils import StopOnAnswer, extract_answer_tail


def generate_cot(tokenizer, model, question, alternatives, max_new_tokens=768, num_return_sequences=1, temperature=0.0, top_p=0.95):


  prompt_content = f"{Instructions}\n\nScenario:\n{question} \n\n Options:\n{alternatives}"
  
  messages = [{"role": "user", "content": prompt_content}]

 
  chat_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
  prompt_ids = chat_inputs["input_ids"]

  decoded_tail = tokenizer.decode(prompt_ids[0][-6:], skip_special_tokens=False)
  if "<think>" not in decoded_tail:
        think_open_ids = tokenizer(
            "<think>\n", add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(model.device)
        prompt_ids = torch.cat([prompt_ids, think_open_ids], dim=1)

  attention_mask = torch.ones_like(prompt_ids)
  prompt_len = prompt_ids.shape[1]

  stopping_criteria = StoppingCriteriaList([StopOnAnswer(tokenizer, prompt_len)])

  # Stage 1: generates reasoning 
  out1 = model.generate(
        input_ids=prompt_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=stopping_criteria,
        repetition_penalty=1.1,
    )

  reasoning_raw = tokenizer.decode(out1[0][prompt_len:], skip_special_tokens=True)
  m = re.search(r"</think>", reasoning_raw, re.IGNORECASE)
  reasoning_text = reasoning_raw[:m.end()] if m else reasoning_raw + "\n</think>"

  final_answer_ids = tokenizer(
        "\nAnswer:", add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(model.device)

  if m:
    prefix_text = reasoning_raw[:m.end()] #reasoning raw correspond to all the generated text by the model in stage 1
    prefix_ids = tokenizer(
           prefix_text, add_special_tokens=False, return_tensors="pt"
       ).input_ids.to(model.device)
    think_end_idx = prompt_len + prefix_ids.shape[1]
    #concatenate "Answer:" to force the model to generate an answer
    stage2_ids = torch.cat([out1[:, :think_end_idx], final_answer_ids], dim=1)
  else:
    stage2_ids = torch.cat([out1, final_answer_ids], dim=1)

  attention_mask2 = torch.ones_like(stage2_ids)
  prompt_len2 = stage2_ids.shape[1]

  # Stage 2: generates the answer, enought tokens for only a number/letter
  out2 = model.generate(
        input_ids=stage2_ids,
        attention_mask=attention_mask2,
        max_new_tokens=64,
        do_sample=False, #deterministically (greedy decoding)
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1,
    )

  answer_tail = tokenizer.decode(out2[0][prompt_len2:], skip_special_tokens=True)
  content = extract_answer_tail(answer_tail)

  generated_text = reasoning_text + "\nAnswer: " + content

  return prompt_content, [generated_text]

### LRM (Large Reasoning Model) Loading

Selects the reasoning model to evaluate (e.g. DeepSeek-R1-Distill or Qwen3) and loads the tokenizer/model onto GPU for inference.

In [6]:
# MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
MODEL_NAME = "Qwen/Qwen3-8B"

In [7]:
# Loading the model and the tokenizer
def load_model(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        attn_implementation="eager",  
        device_map = "cuda:0"
    )
    model.eval()
    return tokenizer, model

In [8]:
tokenizer, model = load_model()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:04<00:00, 83.66it/s]


In [11]:
from utils.datasets import load_sla_violation, extract_sla_violation
from utils.utils import extract_reasoning_and_answer, split_into_sentences, save_cot_relevance_txt, extract_reasoning_and_answer
from utils.kl_divergency import plot_flow_heatmap, plot_sentence_relevance
from utils.entropy import plot_entropy_suppression_delta, compute_predictive_entropy_curve, plot_predictive_entropy_curve
from utils.gradient_input import compute_gradient_x_input_flow, plot_gradient_input_heatmap
from utils.perplexity import compute_conditional_perplexity_flow, plot_conditional_perplexity_heatmap
from utils.integrated_gradient import compute_ig_flow_matrix, plot_ig_heatmap
from utils.analyze import analyze_sentence_flows, rank_sentence_relevance
from utils.segmentation import segment_with_llm

In [14]:
DATASET = "SLA_violation"

dataset_examples  = load_sla_violation(num_samples=1, start=0 )
extract_qa_fn = extract_sla_violation

for example in dataset_examples:
    question, options, answer = extract_sla_violation(example)

    print(f"\nTask: {question!r}\n")
    print(f"Answer: {answer!r}\n")

    prompt_text, generated_texts = generate_cot(
            tokenizer,
            model,
            question,
            alternatives = options,
            max_new_tokens=4096,   
            num_return_sequences=1,
            temperature=0.5,
            top_p=0.95,
        )

    for path_idx, generated_text in enumerate(generated_texts, start=1):
        # path_idx: index of this reasoning attempt when generating multiple paths per question. 
        # Its good for compare different generations.
        print(f"--- Reasoning path {path_idx} ---")
        print(generated_text)
        print("------------------\n")

        reasoning_text, plan_chunk = extract_reasoning_and_answer(generated_text)
        generated_episodes = split_into_sentences(reasoning_text)

        for step in generated_episodes:
            print(f"  {step}")


[{'episode_id': '6gb_prompt_dbaf3a7a3a176522', 'task_id': 'T10', 'task_name': 'SLA Violation Prediction', 'source_turn': 10, 'question': 'The UAV is descending under URLLC slice with latency 8ms, jitter 1.2ms, loss 0.06%, throughput 100Mbps, and edge load 0.3. Historical trends show latency increased from 6ms to 8ms over the last 3 turns, jitter from 0.8ms to 1.2ms, and edge load from 0.25 to 0.3. The SLA requires latency ≤10ms, jitter ≤2ms, loss ≤0.1%, and edge load ≤0.4. If weather-induced degradation continues at the same rate, latency will reach 10ms in 2 turns, jitter 1.6ms, and edge load 0.35. A sudden wind gust may increase latency by up to 3ms and edge load by 0.1 in the next turn. Option A maintains URLLC with current settings. Option B switches to eMBB with 14ms latency, 3ms jitter, 0.2% loss, 800Mbps throughput, and 0.5 edge load. Option C switches to mMTC with 50ms latency, 10ms jitter, 1.5% loss, 1Mbps throughput, and 0.75 edge load. Option D delays descent for 1 turn to c